# wf_STG_Ledger_Validation

Generated from `router_validation_repository.xml!Finance/wf_STG_Ledger_Validation` by compiler 1.4.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import functions as F
def require_transform(name):
    binding = transform_bindings.get(name)
    if binding is None:
        raise ValueError(f"Configure transform_bindings[{name!r}] before running")
    return binding
def _read_source(location):
    if location.endswith(".parquet") or "/parquet/" in location.lower():
        return spark.read.parquet(location)
    if "/" not in location and "\\" not in location and not location.endswith(".csv"):
        return spark.read.table(location)
    return spark.read.option("header", True).option("inferSchema", True).csv(location)


## Source configuration

Replace each empty value with an accessible Fabric or OneLake path before running the ingestion cells.

| Tool ID | Generated DataFrame |
| --- | --- |
| `Ledger_Src_File` | `ledger_src_file` |


In [ ]:
source_paths = {
    "Ledger_Src_File": "",  # ledger_src_file
}


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `ledger_src_file_path`, `ledger_src_file`.


In [ ]:
ledger_src_file_path = source_paths.get('Ledger_Src_File')
if not ledger_src_file_path: raise ValueError("Configure source_paths['Ledger_Src_File'] before running")
ledger_src_file = _read_source(ledger_src_file_path)


## Transformations

Applies the translated Informatica PowerCenter operations in dependency order.


### Transformations 1 of 2

Produces `transform_bindings`, `sq_ledger_src_file`, `exp_date_validation`, `rtr_ledger_routes`, and 4 more.


In [ ]:
transform_bindings = globals().get('transform_bindings', {})
sq_ledger_src_file = ledger_src_file
exp_date_validation = sq_ledger_src_file.withColumn("File_Name", F.trim(F.col("CurrentlyProcessedFileName"))).withColumn("CHANNEL", F.when(F.col("AccountGroup") == F.lit("NORTH_REGION"), F.lit("NORTH")).otherwise(F.lit("SOUTH"))).withColumn("REJECT_REASON", F.when(F.col("AccountGroup").isNull(), F.lit("Missing AccountGroup")).otherwise(F.lit(None)))
rtr_ledger_routes = require_transform("RTR_Ledger")(exp_date_validation)
rtr_ledger_stg_data = rtr_ledger_routes["STG_DATA"]
rtr_ledger_north_rejected = rtr_ledger_routes["NORTH_REJECTED"]
rtr_ledger_south_rejected = rtr_ledger_routes["SOUTH_REJECTED"]
rtr_ledger_default1 = rtr_ledger_routes["DEFAULT1"]


### Transformations 2 of 2

Produces `agg_north_rejects`, `agg_south_rejects`, `stg_ledger_content`, `unprocessed_north_file_list`, and 1 more.


In [ ]:
agg_north_rejects = rtr_ledger_north_rejected.select("File_Name3", "REJECT_REASON3").dropDuplicates()
agg_south_rejects = rtr_ledger_south_rejected.select("File_Name4", "REJECT_REASON4").dropDuplicates()
stg_ledger_content = rtr_ledger_stg_data
unprocessed_north_file_list = agg_north_rejects
unprocessed_south_file_list = agg_south_rejects


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(stg_ledger_content)
display(unprocessed_north_file_list)
display(unprocessed_south_file_list)
